# Andromeda Galaxy Telescope Data Updating Video Example

This notebook creates a **telescope-style updating video** for the Andromeda Galaxy (M31).

It has two modes:

1. **Synthetic demo mode** — works offline and simulates FITS-like exposures, noise, stacking, seeing, and a live telescope HUD.
2. **Local telescope data mode** — replace the synthetic `data_cube` with your own calibrated FITS/image frames and reuse the same stacking/video renderer.

Outputs:

- `andromeda_telescope_preview.png`
- `andromeda_telescope_data_updating_example.mp4`

Note: the included sample is a simulation for education/demo use, not scientific measurement data.


In [ ]:
# Uncomment in a fresh environment if needed:
# %pip install -U numpy pillow imageio imageio-ffmpeg


In [1]:
from __future__ import annotations

from pathlib import Path
import math
import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageFilter
import imageio.v2 as imageio

OUT_DIR = Path('.')
VIDEO_PATH = OUT_DIR / 'andromeda_telescope_data_updating_example.mp4'
PNG_PATH = OUT_DIR / 'andromeda_telescope_preview.png'

W, H = 1280, 720
FPS = 18
SECONDS = 8
NFRAMES = FPS * SECONDS
SEED = 31
rng = np.random.default_rng(SEED)

print('Ready')

Ready


In [2]:
def robust_scale(arr, lo=0.25, hi=99.85, gamma=0.72):
    arr = np.asarray(arr, dtype=np.float32)
    a, b = np.percentile(arr, [lo, hi])
    scaled = np.clip((arr - a) / max(b - a, 1e-6), 0, 1)
    return scaled ** gamma


def make_starfield(width=W, height=H, seed=SEED):
    rng = np.random.default_rng(seed)
    img = np.zeros((height, width), dtype=np.float32)
    yy_full, xx_full = np.mgrid[0:height, 0:width]
    for x, y, f, s in zip(
        rng.uniform(0, width, 1700),
        rng.uniform(0, height, 1700),
        0.08 + 1.65 * (1 - rng.power(3.0, 1700)) ** 2,
        rng.uniform(0.45, 1.25, 1700),
    ):
        r = int(max(2, math.ceil(4 * s)))
        x0, x1 = max(0, int(x)-r), min(width, int(x)+r+1)
        y0, y1 = max(0, int(y)-r), min(height, int(y)+r+1)
        xx = xx_full[y0:y1, x0:x1]
        yy = yy_full[y0:y1, x0:x1]
        img[y0:y1, x0:x1] += f * np.exp(-((xx-x)**2 + (yy-y)**2) / (2*s*s))
    return img


def make_andromeda_base(width=W, height=H, seed=SEED):
    rng = np.random.default_rng(seed)
    y, x = np.mgrid[0:height, 0:width]
    cx, cy = width * 0.50, height * 0.49
    theta = np.deg2rad(-13)
    xr = (x - cx) * np.cos(theta) - (y - cy) * np.sin(theta)
    yr = (x - cx) * np.sin(theta) + (y - cy) * np.cos(theta)
    r = np.sqrt((xr/(width*0.42))**2 + (yr/(height*0.105))**2)

    disk = np.exp(-2.7 * r)
    bulge = 2.4 * np.exp(-((xr/(width*0.085))**2 + (yr/(height*0.06))**2))
    halo = 0.28 * np.exp(-1.25 * r)
    arms = 0.16 * np.cos(7.0 * np.arctan2(yr/(height*0.105), xr/(width*0.42)) + 9.5 * r) * np.exp(-2.0 * r)
    dust = (
        np.exp(-((yr + height*0.040)/(height*0.017))**2) * np.exp(-(xr/(width*0.33))**2)
        + 0.75 * np.exp(-((yr + height*0.078)/(height*0.022))**2) * np.exp(-(xr/(width*0.40))**2)
        + 0.35 * np.exp(-((yr - height*0.055)/(height*0.019))**2) * np.exp(-(xr/(width*0.25))**2)
    )
    galaxy = np.clip(disk + bulge + halo + arms, 0, None) * np.clip(1 - 0.55 * dust, 0.25, 1.0)
    for sx, sy, amp, sxscale, syscale in [(width*0.66, height*0.38, 0.28, 0.035, 0.028), (width*0.34, height*0.66, 0.22, 0.055, 0.040)]:
        galaxy += amp * np.exp(-(((x-sx)/(width*sxscale))**2 + ((y-sy)/(height*syscale))**2))

    stars = make_starfield(width, height, seed+5)
    bg_gradient = 0.018 + 0.025 * (x / width) + 0.012 * (1 - y/height)
    mono = galaxy * 1.3 + stars * 0.6 + bg_gradient

    norm = robust_scale(mono, 0.2, 99.92, gamma=0.78)
    gal_norm = robust_scale(galaxy, 0.1, 99.7, gamma=0.75)
    star_norm = robust_scale(stars, 75, 99.98, gamma=0.65)
    blue_knots = np.maximum(0, arms) * 1.8 + 0.15 * rng.random((height, width)) * (galaxy > 0.07)
    rgb = np.zeros((height, width, 3), dtype=np.float32)
    rgb[..., 0] = 0.58 * norm + 0.37 * gal_norm + 0.10 * star_norm
    rgb[..., 1] = 0.54 * norm + 0.28 * gal_norm + 0.12 * star_norm
    rgb[..., 2] = 0.62 * norm + 0.11 * gal_norm + 0.26 * robust_scale(blue_knots, 40, 99.9, 0.85) + 0.11 * star_norm
    rr = np.sqrt(((x-width/2)/(width/2))**2 + ((y-height/2)/(height/2))**2)
    rgb *= np.clip(1 - 0.45 * rr**1.8, 0.35, 1.0)[..., None]
    return np.clip(rgb, 0, 1)

In [5]:
def add_overlay(frame_rgb, frame_index, total_frames, width=W, height=H):
    im = Image.fromarray(np.clip(frame_rgb * 255, 0, 255).astype(np.uint8), 'RGB')
    draw = ImageDraw.Draw(im, 'RGBA')
    try:
        font_big = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf', 30)
        font = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 20)
        font_small = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf', 16)
        font_mono = ImageFont.truetype('/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf', 18)
    except Exception:
        font_big = font = font_small = font_mono = None
    n_exposures = 1 + int((frame_index / max(total_frames-1, 1)) * 96)
    total_seconds = n_exposures * 45
    snr = math.sqrt(n_exposures) * 8.5
    seeing = 1.42 + 0.10 * math.sin(frame_index * 0.16)
    sky = 21.2 + 0.15 * math.sin(frame_index * 0.11 + 1.0)

    draw.rectangle([0, 0, width, 64], fill=(3, 7, 16, 145))
    draw.text((24, 16), 'M31 / ANDROMEDA — TELESCOPE DATA LIVE STACK', fill=(234, 238, 247, 245), font=font_big)
    draw.text((24, 52), 'Simulated FITS-like exposures: luminance + color composite, updating stack', fill=(190, 203, 218, 225), font=font_small)
    panel_w = 370
    draw.rounded_rectangle([width-panel_w-24, 86, width-24, 260], radius=12, fill=(4, 10, 20, 150), outline=(170,190,220,110), width=1)
    rows = [
        ('Target', 'Andromeda Galaxy / M31'),
        ('RA / Dec', '00h42m44s  +41°16′09″'),
        ('Frame', f'{frame_index+1:03d}/{total_frames:03d}'),
        ('Stacked', f'{n_exposures:03d} × 45 s  ({total_seconds/60:4.1f} min)'),
        ('Seeing', f'{seeing:0.2f} arcsec'),
        ('Sky', f'{sky:0.2f} mag/arcsec²'),
        ('SNR', f'{snr:0.1f}')]
    y0 = 102
    for k, v in rows:
        draw.text((width-panel_w-6, y0), k, fill=(156, 178, 204, 230), font=font_small)
        draw.text((width-panel_w+112, y0), v, fill=(232, 238, 248, 245), font=font_mono)
        y0 += 22
    p = frame_index/(total_frames-1)
    bx0, by0, bx1, by1 = 24, height-48, width-24, height-25
    draw.rounded_rectangle([bx0, by0, bx1, by1], radius=8, fill=(5, 10, 18, 165), outline=(170,190,220,100), width=1)
    draw.rounded_rectangle([bx0+3, by0+3, bx0+3+int((bx1-bx0-6)*p), by1-3], radius=6, fill=(220, 230, 250, 165))
    draw.text((24, height-78), 'Updating stacked telescope frames: noise drops as more exposures are added', fill=(220, 228, 240, 230), font=font_small)
    sweep_x = int(width * ((frame_index % FPS) / FPS))
    draw.rectangle([sweep_x, 64, min(width, sweep_x+3), height-86], fill=(210, 230, 255, 45))
    return np.asarray(im)

In [6]:
base = make_andromeda_base()
Image.fromarray((base*255).astype(np.uint8)).save(PNG_PATH)

writer = imageio.get_writer(
    VIDEO_PATH,
    fps=FPS,
    codec='libx264',
    quality=8,
    macro_block_size=16,
    output_params=['-pix_fmt', 'yuv420p', '-crf', '24']
)
try:
    for i in range(NFRAMES):
        p = i/(NFRAMES-1)
        nstack = 1 + int(p * 96)
        noise_sigma = 0.070 / math.sqrt(nstack) + 0.004
        frame = np.clip(base + rng.normal(0, noise_sigma, base.shape).astype(np.float32), 0, 1)
        frame = np.clip((frame - 0.02) * (0.92 + 0.16*p), 0, 1)
        img = Image.fromarray((frame*255).astype(np.uint8), 'RGB')
        zoom = 1.0 + 0.055*p
        crop_w, crop_h = int(W / zoom), int(H / zoom)
        pan_x = int((W-crop_w) * (0.50 + 0.25*math.sin(2*math.pi*p)))
        pan_y = int((H-crop_h) * (0.52 + 0.18*math.cos(2*math.pi*p)))
        crop = img.crop((pan_x, pan_y, pan_x+crop_w, pan_y+crop_h)).resize((W, H), Image.Resampling.LANCZOS)
        frame = np.asarray(crop).astype(np.float32)/255.0
        writer.append_data(add_overlay(frame, i, NFRAMES))
finally:
    writer.close()

print('Saved:', PNG_PATH.resolve())
print('Saved:', VIDEO_PATH.resolve())

Multiple -pix_fmt options specified for stream 0, only the last option '-pix_fmt yuv420p' will be used.


Saved: /home/jatin/Desktop/andromeda_telescope_preview.png
Saved: /home/jatin/Desktop/andromeda_telescope_data_updating_example.mp4


## Optional: use real telescope/FITS frames instead of the synthetic galaxy

To adapt this for real telescope data, create a 3D NumPy array named `data_cube` with shape:

```python
(number_of_exposures, height, width)
```

Each slice should be a calibrated light frame, ideally after bias/dark/flat correction and star alignment. Then replace the synthetic `base + noise` section in the render loop with a running stack, for example:

```python
stack = np.mean(data_cube[:nstack], axis=0)
frame = robust_scale(stack)
frame_rgb = np.dstack([frame, frame, frame])
writer.append_data(add_overlay(frame_rgb, i, NFRAMES))
```

For FITS files, install Astropy and read them with:

```python
from astropy.io import fits
frame = fits.getdata('your_m31_frame.fits').astype('float32')
```
